In [ ]:
%pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24888.80it/s]


torch.Size([4, 4])


In [2]:
import json
with open("datas/train.jsonl", "r") as f:
    data_train = [json.loads(line) for line in f]


with open("datas/test.jsonl", "r") as f:
    data_test = [json.loads(line) for line in f]

with open("datas/devel.jsonl", "r") as f:
    data_devel = [json.loads(line) for line in f]

In [1]:
import pandas as pd
train_df = pd.DataFrame(data_train)
test_df = pd.DataFrame(data_test)

NameError: name 'data_train' is not defined

In [16]:
train_df["intent"]

0           calendar_set
1           calendar_set
2        audio_volume_up
3        iot_hue_lightup
4          weather_query
              ...       
11509     takeaway_query
11510     datetime_query
11511        email_query
11512     play_audiobook
11513       calendar_set
Name: intent, Length: 11514, dtype: str

In [17]:
train_df["intent"].value_counts()

intent
calendar_set      804
play_music        620
weather_query     559
calendar_query    558
general_quirky    546
                 ... 
events              1
convert             1
cleaning            1
settings            1
volume_other        1
Name: count, Length: 91, dtype: int64

In [27]:
df = pd.concat([train_df[["sentence", "intent"]],test_df[["sentence", "intent"]], pd.DataFrame(data_devel)[["sentence", "intent"]]], axis=0, ignore_index=True)

In [28]:
df = df.groupby("intent").filter(lambda x: len(x) >= 2)

In [29]:
df.describe(include="all")

,sentence,intent
count,16510,16510
unique,16486,82
top,do i have any new email,calendar_set
freq,3,1142


In [30]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

In [31]:
df

,sentence,intent
0,event,calendar_set
1,i need an event three days from now scheduled ...,calendar_set
2,turn up,audio_volume_up
3,brighten the lights a little bit,iot_hue_lightup
4,what's the current weather,weather_query
...,...,...
16516,can you give me local news on wayne county she...,news_query
16517,every light of room increase its intensity,iot_hue_lightup
16518,i would like some coffee now,iot_coffee
16519,what is the population of los angeles,qa_factoid


In [48]:
df = (
    df.groupby("intent", group_keys=False).filter(lambda x: len(x) >= 5).groupby("intent", group_keys=False).sample(n=5, random_state=42))

In [49]:
df["intent"].value_counts()

intent
alarm_query          5
alarm_remove         5
alarm_set            5
audio_volume_down    5
audio_volume_mute    5
                    ..
transport_query      5
transport_taxi       5
transport_ticket     5
transport_traffic    5
weather_query        5
Name: count, Length: 75, dtype: int64

In [7]:
df.to_json("datas/df_slurp.jsonl", orient="records", lines=True, force_ascii=False)

NameError: name 'df' is not defined

In [9]:
import pandas as pd

In [10]:
df = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)

In [11]:
df["intent"].value_counts()

intent
alarm_query          12
alarm_remove         12
alarm_set            12
audio_volume_down    12
audio_volume_mute    12
                     ..
transport_taxi       12
transport_ticket     12
transport_traffic    12
weather_query        12
wemo_off             12
Name: count, Length: 82, dtype: int64

In [12]:
import numpy as np
df_test = pd.read_json("datas/df_test.jsonl", orient="records", lines=True)
df_train = pd.read_json("datas/df_train.jsonl", orient="records", lines=True)
intents = sorted(df_train["intent"].unique())
example_vectors = model.encode(
    df["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=False
)
action_vectors = np.stack([
    example_vectors[df["intent"].to_numpy() == intent].mean(axis=0)
    for intent in intents
])
action_vectors /= np.linalg.norm(action_vectors, axis=1, keepdims=True)
print(f"{len(intents)} интентов")

NameError: name 'model' is not defined

In [13]:
def prepare_model(model):
    example_vectors = model.encode(
        df["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    action_vectors = np.stack([
        example_vectors[df["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents
    ])
    action_vectors /= np.linalg.norm(action_vectors, axis=1, keepdims=True)
    return action_vectors

In [15]:
df_test

,sentence,intent
0,Какие будильники у меня включены?,alarm_query
1,Қандай оятқыштарым қосулы тұр?,alarm_query
2,Which alarms are currently active?,alarm_query
3,Удали будильник на шесть утра,alarm_remove
4,Таңғы алтыдағы оятқышты өшірші,alarm_remove
...,...,...
241,Демалыс күндеріне ауа райы болжамы қандай?,weather_query
242,What is the weekend forecast?,weather_query
243,Отключи умную розетку у лампы,wemo_off
244,Шам жанындағы ақылды розетканы ажырат,wemo_off


In [16]:
from sklearn.metrics import accuracy_score, f1_score
def test_model(model, action_vectors_for_model):
    assert set(df_test["intent"]) == set(intents)
    assert df_test["intent"].value_counts().eq(3).all()

    test_vectors = model.encode(
        df_test["sentence"].tolist(), normalize_embeddings=True, show_progress_bar=True
    )
    similarities = test_vectors @ action_vectors_for_model.T
    best_indices = np.argmax(similarities, axis=1)
    best_scores = np.clip(similarities[np.arange(len(best_indices)), best_indices], 0, 1)
    y_pred = np.asarray(intents, dtype=object)[best_indices]
    y_pred[best_scores < THRESHOLD] = "unknown"
    y_true = df_test["intent"].to_numpy()

    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1_macro = f1_score(y_true, y_pred, labels=intents, average="macro", zero_division=0)
    print(f"Test accuracy: {test_accuracy}")
    print(f"Test macro F1: {test_f1_macro}")
    assert all(df_test["intent"].iloc[i:i + 3].nunique() == 1 for i in range(0, len(df_test), 3))
    for offset, language in enumerate(("Russian", "Kazakh", "English")):
        mask = np.arange(len(df_test)) % 3 == offset
        language_accuracy = accuracy_score(y_true[mask], y_pred[mask])
        print(f"{language} accuracy: {language_accuracy:.4f}")

In [17]:
def test_pipe(model):
    action_vectors_for_model = prepare_model(model)
    test_model(model, action_vectors_for_model)

In [18]:
import pandas as pd

df_small = pd.read_json("datas/df_small.jsonl", orient="records", lines=True)
df_small_train = pd.read_json("datas/df_small_train.jsonl", orient="records", lines=True)
df_small_test = pd.read_json("datas/df_small_test.jsonl", orient="records", lines=True)
print(f"Small dataset: {len(df_small)} examples, "
      f"{len(df_small_train)} train, {len(df_small_test)} test, "
      f"{df_small['intent'].nunique()} intents")


Small dataset: 120 examples, 96 train, 24 test, 8 intents


In [19]:
def test_pipe_small(model, threshold=0.60):
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score

    intents_small = sorted(df_small_train["intent"].unique())
    train_vectors = model.encode(df_small_train["sentence"].tolist(), normalize_embeddings=True)
    centers = np.stack([
        train_vectors[df_small_train["intent"].to_numpy() == intent].mean(axis=0)
        for intent in intents_small
    ])
    centers /= np.linalg.norm(centers, axis=1, keepdims=True)

    test_vectors = model.encode(df_small_test["sentence"].tolist(), normalize_embeddings=True)
    scores = test_vectors @ centers.T
    predictions = np.array(intents_small, dtype=object)[scores.argmax(axis=1)]
    predictions[scores.max(axis=1) < threshold] = "unknown"
    answers = df_small_test["intent"].to_numpy()

    print(f"Test accuracy: {accuracy_score(answers, predictions):.4f}")
    print(f"Test macro F1: {f1_score(answers, predictions, labels=intents_small, average='macro', zero_division=0):.4f}")
    for offset, language in enumerate(("Russian", "Kazakh", "English")):
        indices = np.arange(len(answers)) % 3 == offset
        print(f"{language} accuracy: {accuracy_score(answers[indices], predictions[indices]):.4f}")


In [1]:
from sentence_transformers import SentenceTransformer

/home/alexseyka/.venvs/ml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:

print("BAAI/bge-m3")
model = SentenceTransformer("BAAI/bge-m3")
test_pipe(model=model)


BAAI/bge-m3


Batches: 100%|██████████| 8/8 [00:30<00:00,  3.76s/it]

Test accuracy: 0.6056910569105691
Test macro F1: 0.5817627494456763
Russian accuracy: 0.6341
Kazakh accuracy: 0.5610
English accuracy: 0.6220


In [37]:
test_pipe_small(model=model, threshold=0.40)

Test accuracy: 1.0000
Test macro F1: 1.0000
Russian accuracy: 1.0000
Kazakh accuracy: 1.0000
English accuracy: 1.0000


In [30]:
model_multilinguale5s = SentenceTransformer("intfloat/multilingual-e5-small")
print("Multilingual E5 small 0.1B; dimension 384; 12 layers")
test_pipe(model=model_multilinguale5s)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4687.58it/s]


Multilingual E5 small 0.1B; dimension 384; 12 layers


Batches: 100%|██████████| 8/8 [00:05<00:00,  1.59it/s]

Test accuracy: 0.4065040650406504
Test macro F1: 0.3857336430507162
Russian accuracy: 0.3780
Kazakh accuracy: 0.3659
English accuracy: 0.4756


In [38]:
model_multilinguale5b = SentenceTransformer("intfloat/multilingual-e5-base")
print("Multilingual E5 base 0.3B; dimension 768; 12 layers")
test_pipe(model=model_multilinguale5b)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1093.49it/s]


Multilingual E5 base 0.3B; dimension 768; 12 layers


Batches: 100%|██████████| 8/8 [00:11<00:00,  1.44s/it]

Test accuracy: 0.524390243902439
Test macro F1: 0.484032498666645
Russian accuracy: 0.5244
Kazakh accuracy: 0.4756
English accuracy: 0.5732


In [39]:
model_multilinguale5l = SentenceTransformer("intfloat/multilingual-e5-large")
print("Multilingual E5 large 0.6B; dimension 1024; 12 layers")
test_pipe(model=model_multilinguale5l)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3113.65it/s]


Multilingual E5 large 0.6B; dimension 1024; 12 layers


Batches: 100%|██████████| 8/8 [00:27<00:00,  3.47s/it]

Test accuracy: 0.5772357723577236
Test macro F1: 0.5534456058846302
Russian accuracy: 0.6220
Kazakh accuracy: 0.4756
English accuracy: 0.6341


In [40]:
model_hiveurban = SentenceTransformer("Hiveurban/multilingual-e5-large-pooled")
print("Multilingual E5 large pooled 0.6B; dimension 1024; 12 layers")
test_pipe(model=model_hiveurban)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5548.87it/s]


Multilingual E5 large pooled 0.6B; dimension 1024; 12 layers


Batches: 100%|██████████| 8/8 [00:29<00:00,  3.74s/it]

Test accuracy: 0.5772357723577236
Test macro F1: 0.5534456058846302
Russian accuracy: 0.6220
Kazakh accuracy: 0.4756
English accuracy: 0.6341


In [41]:
model_darmm = SentenceTransformer("Darmm/darmm-embed-kazakh-v2")
print("Darmm/darmm-embed-kazakh-v2 0.6B; dimension 1024; 12 layers")
test_pipe(model=model_darmm)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5104.02it/s]


Darmm/darmm-embed-kazakh-v2 0.6B; dimension 1024; 12 layers


Batches: 100%|██████████| 8/8 [00:25<00:00,  3.20s/it]

Test accuracy: 0.36585365853658536
Test macro F1: 0.34956094041459895
Russian accuracy: 0.3049
Kazakh accuracy: 0.3415
English accuracy: 0.4512


In [20]:
model_multilinguale5linstruct = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
print("Multilingual E5 large instruct 0.6B; dimension 1024; 12 layers")
test_pipe(model=model_multilinguale5linstruct)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6361.29it/s]


Multilingual E5 large instruct 0.6B; dimension 1024; 12 layers


Batches: 100%|██████████| 8/8 [00:26<00:00,  3.34s/it]

Test accuracy: 0.556910569105691
Test macro F1: 0.5335564841662401
Russian accuracy: 0.5854
Kazakh accuracy: 0.4390
English accuracy: 0.6463


In [21]:
model_nomic = SentenceTransformer("nomic-ai/nomic-embed-text-v2-moe")
print("Nomic Embed Text v2 MoE 0.5B; activ 0.3B")
test_pipe(model=model_nomic)

Loading weights: 100%|██████████| 82/82 [00:00<00:00, 9024.74it/s]
[transformers] NomicBertModel LOAD REPORT from: nomic-ai/nomic-embed-text-v2-moe
Key                                                | Status     | 
---------------------------------------------------+------------+-
layers.{0...11}.self_attn.k_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w1      | UNEXPECTED | 
layers.{0...11}.self_attn.o_proj.bias              | UNEXPECTED | 
layers.{0...11}.self_attn.q_proj.bias              | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.down_proj.bias      | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.bias        | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.experts.mlp.w2      | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.weight          | UNEXPECTED | 
layers.{0...11}.self_attn.v_proj.bias              | UNEXPECTED | 
layers.{1, 3, 5, 7, 9, 11}.mlp.router.layer.weight | UNEXPECTED | 
layers.{0, 2, 4, 6, 8, 10}.mlp.fc1.bias         

Nomic Embed Text v2 MoE 0.5B; activ 0.3B


Batches: 100%|██████████| 8/8 [00:14<00:00,  1.79s/it]

Test accuracy: 0.15040650406504066
Test macro F1: 0.1409504452187379
Russian accuracy: 0.0976
Kazakh accuracy: 0.1585
English accuracy: 0.1951


In [24]:
model_qwen = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Qwen3 Embedding 0.6B")
test_pipe(model=model_qwen)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7118.37it/s]


Qwen3 Embedding 0.6B


Batches: 100%|██████████| 8/8 [01:19<00:00,  9.93s/it]

Test accuracy: 0.43902439024390244
Test macro F1: 0.4454413472706156
Russian accuracy: 0.4756
Kazakh accuracy: 0.2439
English accuracy: 0.5976


In [ ]:
def predict(query, model):
    action_vectors_for_model = prepare_model(model)

    query_vector = model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    )[0]
    similarities = action_vectors_for_model @ query_vector

    best = int(np.argmax(similarities))
    confidence = float(np.clip(similarities[best], 0, 1))

    return {
        "endpoint": intents[best] if confidence >= THRESHOLD else "unknown",
        "confidence": round(confidence, 4),
    }

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import time
app = FastAPI()
class QueryRequest(BaseModel):
    query: str
@app.post('/classify')
def classify(request: QueryRequest):
    try:
        start_time = time.time()
        result = predict(request.query)
        end_time = time.time()
        print(f"Query processing time: {end_time - start_time:.2f} seconds")
        return result
    except ValueError as error:
        raise HTTPException(status_code=422, detail=str(error)) from error

from fastapi.testclient import TestClient
with TestClient(app) as client:
    print(client.post('/classify', json={'query': 'four am'}).json())

In [ ]:
async def serve():
    import uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000))
    await server.serve()